## 💻 SFT for Code Suggestion
*Experiment: Best run*

---

### 📌 Key Parameters
*   **Rank:** `32`
*   **Alpha:** `32`
*   **Target Gates:** `q`, `k`, `v`, `o`, `gate`, `up`, `down`
*   **Epochs:** `2`
*   **Learning Rate:** `5e-4`

In [ ]:
%cd ..

/workspace/code-suggester


## 1) Import Libraries

In [ ]:
# Check GPU
!nvidia-smi


Thu Oct  1 16:43:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   39C    P8             34W /  340W |       7MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import unsloth
import os
import gc
from pathlib import Path

import torch
from unsloth import FastLanguageModel
from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, DatasetDict
from trl import SFTTrainer, SFTConfig

from huggingface_hub import login, HfApi, upload_folder
import getpass
import datetime

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/workspace/code-suggester/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1001 16:43:29.481000 2774 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1001 16:43:29.533000 2774 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/workspace/code-suggester/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:3525: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


## 2) Config (model + data + hyperparams)


In [ ]:
set_seed(42)

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_NAME = "jiacheng-ye/nl2bash"
SYSTEM_PROMPT = """
You are an expert Linux command-line assistant. Your sole task is to translate natural language requests into precise, executable Bash commands.

Strict Output Rules:
1. Output ONLY the raw Bash command.
2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```bash ... ```).
3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the command:" or "Sure!").
4. Chain commands logically using pipes (|), &&, or || where appropriate.
"""


# ====== TRAINING ======
MAX_SEQ_LENGTH = 2048
RANK = 32
ALPHA_RANK = 32
GATES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
NUM_EPOCHS = 2
LR = 5e-4
WARMUP_STEPS = 100

# Batch
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8

# Outputs
WORKDIR = Path.cwd().resolve()
OUTPUT_DIR = WORKDIR / "artifacts" / "qwen2.5-bash-7B-best-lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Memory
LOAD_IN_4BIT = False

# dtype
DTYPE = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("BASE_MODEL:", BASE_MODEL)
print("DTYPE:", DTYPE, "| LOAD_IN_4BIT:", LOAD_IN_4BIT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


BASE_MODEL: Qwen/Qwen2.5-Coder-7B-Instruct
DTYPE: bfloat16 | LOAD_IN_4BIT: False
GPU: NVIDIA GeForce RTX 3090


## 3) Load model + Turn on LoRA (Unsloth)


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = RANK,
    target_modules = GATES,
    lora_alpha = ALPHA_RANK,
    bias = "none",
    use_gradient_checkpointing = True,
    use_rslora = False,
    loftq_config = None,
    random_state=42
)
model.config.use_cache = False
model.print_trainable_parameters()

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print("Loaded model + LoRA OK")


==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.559 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 339/339 [00:04<00:00, 76.19it/s]
Unsloth 2026.9.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491
Loaded model + LoRA OK


## 4) Load dataset JSONL & Format by chat_template


In [ ]:
raw_dataset = load_dataset(DATASET_NAME, revision="refs/convert/parquet")
dataset = DatasetDict({
    "train": raw_dataset["train"],
    "val": raw_dataset["validation"],
    "test": raw_dataset["test"]
})

print(dataset)
sample = dataset["train"][0]
print("\nQuestion preview:\n")
print(sample["nl"][:500])
print("\nAnswer preview:\n")
print(sample["bash"][:500])

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 724
    })
    val: Dataset({
        features: ['messages'],
        num_rows: 90
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 91
    })
})

User Prompt preview:

I need a method that takes a string of text and breaks it down into tokens based on specific special tokens defined in the class. The method should return a list of token IDs. It should handle cases where special tokens are not found in the remaining text and ensure that the text is processed sequentially. Could you help me implement this?

Assistant Code preview:

def tokenize(self, text):
    ids = []
    idx = 0
    while 1:
        indices = {}
        for token in self._special_tokens:
            try:
                indices[token] = text[idx:].index(token)
            except ValueError:
                continue
        if len(indices) == 0:
            break
        next_token = min(indices, key=indices

In [ ]:
def formatting_prompts_func(examples):
    questions = examples["nl"]
    answers = examples["bash"]

    texts = []

    for q, a in zip(questions, answers):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q.strip()},
            {"role": "assistant", "content": a.strip()}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=2)

In [ ]:
print("Dataset size:", len(dataset["train"]))
print("\n===== PREVIEW 2 SAMPLES =====")
for i in range(2):
    print(f"\n--- Sample {i+1} ---\n{dataset['train'][i]['text']}\n...")

Dataset size: 724

===== PREVIEW 2 SAMPLES =====

--- Sample 1 ---
<|im_start|>system

You are an elite AI Backend Engineer. Your sole task is to translate natural language requirements into precise, production-grade Python code for AI systems (e.g., FastAPI, SQLAlchemy, Vector databases, LLM integrations).

Strict Output Rules:
1. Output ONLY the raw Python code.
2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```python ... ``` or ```).
3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the code:" or "Sure!").
4. Enforce production best practices implicitly: utilize async/await for I/O-bound operations, include strict type hints, ensure proper resource management (e.g., context managers for database sessions), and never block the event loop.
<|im_end|>
<|im_start|>user
I need a method that takes a string of text and breaks it down into tokens based on specific special tokens defined in the class. The method sh

## 5) Train SFT


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["val"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,

    args = SFTConfig(
        gradient_accumulation_steps = GRAD_ACCUM,
        per_device_train_batch_size = PER_DEVICE_BATCH,
        per_device_eval_batch_size = PER_DEVICE_BATCH,
        learning_rate = LR,
        num_train_epochs = NUM_EPOCHS,
        warmup_steps = WARMUP_STEPS,

        logging_steps = 20,
        eval_strategy = "steps",
        eval_steps = 20,
        report_to = "none",

        save_strategy = "steps",
        save_steps = 20,
        save_total_limit = 2,

        fp16 = (DTYPE == "float16"),
        bf16 = (DTYPE == "bfloat16"),
        optim = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        output_dir = OUTPUT_DIR,

        completion_only_loss = True,
        gradient_checkpointing=True
    ),
)

train_result = trainer.train()
train_result.metrics

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 724 | Num Epochs = 2 | Total steps = 92
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/workspace/code-suggester/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/workspace/code-suggester/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/workspace/code-suggester/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'

Step,Training Loss,Validation Loss
20,1.681879,1.017350
40,0.766568,0.628110
60,0.606855,0.580603
80,0.577037,0.553815


Unsloth: Restored added_tokens_decoder metadata in /workspace/code-suggester/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/code-suggester/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/code-suggester/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/code-suggester/artifacts/qwen2.5-coder-7B-best-lora/checkpoint-80/tokenizer_config.json.


In [ ]:
for callback in trainer.callback_handler.callbacks.copy():
    if "Notebook" in callback.__class__.__name__:
        trainer.remove_callback(callback)

eval_results = trainer.evaluate()
print("===== VAL SET METRICS =====")
eval_results

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_test_dataset = dataset["test"].map(
    tokenize_function,
    batched=True,
    num_proc=2
)

test_results = trainer.evaluate(
    eval_dataset=tokenized_test_dataset,
    metric_key_prefix="test"
)

print("===== TEST SET METRICS =====")
print(test_results)

## 6) Save LoRA Adapter

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to:", OUTPUT_DIR)

## 7) Push to Hugging Face

In [ ]:
hf_token = getpass.getpass()
login(token=hf_token)
print("HF login OK")

In [ ]:
api = HfApi()
USERNAME = api.whoami()["name"]

REPO = f"{USERNAME}/Qwen2.5-Coder-7B-Instruct-backend-LoRA"
api.create_repo(REPO, private=False, exist_ok=True)

In [ ]:
ckpt_path = "./artifacts/qwen2.5-coder-7B-best-lora"
tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model = AutoModelForCausalLM.from_pretrained(ckpt_path)

In [ ]:
model.push_to_hub(
    REPO,
    commit_message="Retrain model, add ckpt",
)

tokenizer.push_to_hub(
    REPO,
    commit_message="Retrain tokenizer, add ckpt",
)

## 8) Sanity check


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
model.to(DEVICE)
FastLanguageModel.for_inference(model)

user_question = "Write a FastAPI endpoint (async def) that accepts a long piece of text, uses Hugging Face AutoTokenizer to count the number of tokens, and returns the result. The system needs to handle thousands of concurrent requests."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_question}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(DEVICE)

outputs = model.generate(
    **inputs,
    max_new_tokens=3000,
    use_cache=True,
    temperature=0.1,
)

prompt_length = inputs["input_ids"].shape[1]
response_tokens = outputs[0][prompt_length:]
response = tokenizer.decode(response_tokens, skip_special_tokens=True)

print("===== TEST INFERENCE =====")
print(f"User: {user_question}")
print(f"Code: {response.strip()}")